<a href="https://colab.research.google.com/github/viktoriagajdosova/Enhancing-Meta-Research-in-Psychology-by-Generative-AI/blob/main/Enhancing-Meta-Research-in-Psychology-by-Generative-AI/pipelines/01_embedding-for-psychometrics/content_overlap_prompt_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- FACTORS/ITEMS ---
import os
import json
import textwrap
import time
from typing import List, Dict, Any
from google.colab import userdata
import warnings
from openai import OpenAI
from google import genai
from google.genai import types

warnings.filterwarnings('ignore')

class GlobalContentComparator:
    def __init__(self, platform: str, model_name: str):
        self.platform = platform.upper()
        self.model_name = model_name
        self.client = None
        self._initialize_client()

    def _initialize_client(self):
        """Initializes API clients for OpenAI or Gemini."""
        if self.platform == 'OPENAI':
            api_key = userdata.get('OPENAI_API_KEY')
            self.client = OpenAI(api_key=api_key)
            print(f"✅ OpenAI client initialized ({self.model_name}).")
        elif self.platform == 'GEMINI':
            api_key = userdata.get('GOOGLE_API_KEY')
            self.client = genai.Client(api_key=api_key, http_options={'api_version': 'v1alpha'})
            print(f"✅ Gemini client initialized ({self.model_name}).")

    def get_global_overlap(self, questionnaire_a: List[str], questionnaire_b: List[str]) -> Dict[str, Any]:
        """
        Calculates the content overlap between two full sets of items.
        """
        # Strict Schema Definition
        gemini_schema = types.Schema(
            type=types.Type.OBJECT,
            properties={
                "global_similarity_score": types.Schema(type=types.Type.NUMBER),
                "shared_themes_summary": types.Schema(type=types.Type.ARRAY, items=types.Schema(
                    type=types.Type.OBJECT,
                    properties={
                        "theme": types.Schema(type=types.Type.STRING),
                        "description": types.Schema(type=types.Type.STRING),
                        "items_from_a": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING)),
                        "items_from_b": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING))
                    },
                    required=["theme", "description", "items_from_a", "items_from_b"]
                )),
                "unique_concepts_a": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING)),
                "unique_concepts_b": types.Schema(type=types.Type.ARRAY, items=types.Schema(type=types.Type.STRING)),
                "detailed_critique": types.Schema(type=types.Type.STRING)
            },
            required=["global_similarity_score", "shared_themes_summary", "detailed_critique"]
        )

        prompt = textwrap.dedent(f"""
            Role: Conservative and Critical Psychometric Auditor and Content Validity Expert.
            Task: Compare the content overlap between Questionnaire A and Questionnaire B.
            Content overlap refers to the shared, similar, or redundant information present across different texts.
            Do not just look for thematic similarity. Evaluate based on 'Clinical Interchangeability'.
            Your score must reflect the loss of information when moving from one to the other.


            QUESTIONNAIRE A:
            {json.dumps(questionnaire_a, indent=2, ensure_ascii=False)}

            QUESTIONNAIRE B:
            {json.dumps(questionnaire_b, indent=2, ensure_ascii=False)}

            STRICT INSTRUCTIONS:
            1. 'global_similarity_score': A float (0.00 to 1.00) for total content overlap. Be highly discriminatory. If the scales have different item counts or varying levels of symptom depth, the score should reflect that gap significantly.
            2. 'shared_themes_summary': An array of objects.
               - "theme": Category name (e.g., Withdrawal, Tolerance).
               - "description": Provide a full explanatory sentence here.
               - "items_from_a": Exact strings from A matching this theme.
               - "items_from_b": Exact strings from B matching this theme.
            3. 'unique_concepts_a/b': Arrays of items with no equivalent in the other list.
            4. 'detailed_critique': A final summary of the content validity gap.

            OUTPUT ONLY VALID JSON. Account for EVERY item from both questionnaires.
        """)

        try:
            if self.platform == 'OPENAI':
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0,
                    response_format={"type": "json_object"}
                )
                return json.loads(response.choices[0].message.content)
            else:
                config = genai.types.GenerateContentConfig(
                    temperature=0.0,
                    response_mime_type="application/json",
                    response_schema=gemini_schema
                )
                response = self.client.models.generate_content(model=self.model_name, contents=prompt, config=config)
                return json.loads(response.text)
        except Exception as e:
            return {"error": str(e)}

In [2]:
# --- FACTOR/ITEMS2 ---

import pandas as pd
from IPython.display import display, HTML
import textwrap

# --- CONFIGURATION ---
PLATFORM = 'OPENAI'
MODEL = 'gpt-5.1'

#PLATFORM = 'GEMINI'
#MODEL = 'gemini-3-pro-preview'

#PLATFORM = 'OPENAI'
#MODEL = 'gpt-5.1'

# --- QUESTIONNAIRES ---
questionnaire_a = [
"A really bad car, boat, train or airplane accident",
"A really bad accident at work or home",
"A hurricane, flood, earthquake, tornado or fire",
"Hit or kicked hard enough to injure - as a child",
"Hit or kicked hard enough to injure - as an adult",
"Forced or made to have sexual contant - as a child",
"Forced or made to have sexual contant - as an adult",
"Attack with a gun, knife or weapon",
"During military service - seeing something horrible or being badly scared",
"Sudden death of close family or friend",
"Seeing someone die suddenly or get badly hurt or killed",
"Some other suddent event that made you feel very scared, helpless, or horrified",
"Sudden move or loss of home and possessions",
"Suddenly abandoned by spouse, partner, parent or family",
                    ]

questionnaire_b = [
"Natural disasters",
"Motor vehicle accidents",
"Other accidents",
"Warfare or combat",
"Sudden death of close friend or loved one",
"Robbery involving a weapon",
"Severe assault by acquaintance or stranger",
"Witness to severe assault of acquaintance or stranger",
"Threat of death or serious bodily harm",
"Childhood physical abuse",
"Witness to family violence",
"Physical abuse by an intimate partner",
"Sexual abuse before age 13 by someone at least 5 year older",
"Sexual abuse before age 13 by someone close in age",
"Sexual abuse during adolescence",
"Sexual abuse as an adult",
"Stalking",
"Life-threatening illness",
"Life-threatening or permanently disabling event for loved one",
"Miscarriage",
"Abortion",

                    ]

# --- EXECUTION ---
comparator = GlobalContentComparator(PLATFORM, MODEL)
print("⏳ Analyzing content overlap...")
result = comparator.get_global_overlap(questionnaire_a, questionnaire_b)

# --- RESULTS DISPLAY ---
if "error" not in result:
    score = result.get('global_similarity_score', 0)

    # Color mapping based on Evans (1996) classification
    if score < 0.2:
        color, label = "#f44336", "VERY WEAK"
    elif score < 0.4:
        color, label = "#ff9800", "WEAK"
    elif score < 0.6:
        color, label = "#ffc107", "MODERATE"
    elif score < 0.8:
        color, label = "#8bc34a", "STRONG"
    else:
        color, label = "#4caf50", "VERY STRONG"

    # 1. Main Header with Visual Score and Progress Bar
    display(HTML(f"""
        <div style="padding:25px; border-radius:12px; background-color:#ffffff; border: 1px solid #e0e0e0; border-left: 12px solid {color}; box-shadow: 2px 2px 10px rgba(0,0,0,0.05); font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
            <div style="display: flex; justify-content: space-between; align-items: center;">
                <div>
                    <h4 style="margin:0; color:#7f8c8d; text-transform: uppercase; letter-spacing: 1.5px; font-size: 12px;">Jaccard Similarity Index</h4>
                    <h1 style="margin:5px 0; color:#2c3e50; font-size: 36px;">{score:.2f}</h1>
                </div>
                <div style="text-align: right;">
                    <span style="background-color:{color}; color:white; padding:8px 18px; border-radius:25px; font-weight:bold; font-size:13px; letter-spacing: 0.5px;">
                        {label} OVERLAP
                    </span>
                </div>
            </div>
            <div style="width: 100%; background-color: #f0f0f0; border-radius: 10px; margin-top: 18px; height: 10px;">
                <div style="width: {score*100}%; background-color: {color}; height: 10px; border-radius: 10px; transition: width 1s ease-in-out;"></div>
            </div>
        </div>
    """))

    # 2. Shared Themes Table (Enhanced Pandas Styling)
    display(HTML("<h3 style='margin-top:35px; color:#2c3e50; font-family: sans-serif; border-bottom: 2px solid #eee; padding-bottom: 10px;'>✅ Shared Symptoms & Semantic Mapping</h3>"))
    themes_list = result.get('shared_themes_summary', [])
    formatted_themes = []

    for item in themes_list:
        if isinstance(item, dict):
            desc = item.get('description') or item.get('desc') or "Detailed explanation not provided."
            formatted_themes.append({
                "Theme (Symptom Cluster)": f"<b style='color:#2c3e50;'>{item.get('theme', 'Unnamed')}</b>",
                "Clinical Description": desc,
                "Items in Questionnaire A": "• " + "<br>• ".join(item.get('items_from_a', [])) if item.get('items_from_a') else "<span style='color:gray;'>None</span>",
                "Items in Questionnaire B": "• " + "<br>• ".join(item.get('items_from_b', [])) if item.get('items_from_b') else "<span style='color:gray;'>None</span>"
            })

    if formatted_themes:
        df = pd.DataFrame(formatted_themes)
        styled_df = df.style.set_properties(**{
            'text-align': 'left',
            'vertical-align': 'top',
            'padding': '12px',
            'border-bottom': '1px solid #f0f0f0'
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#f8f9fa'), ('color', '#455a64'), ('font-weight', 'bold'), ('text-transform', 'uppercase'), ('font-size', '11px')]}
        ])
        display(HTML(styled_df.to_html(escape=False)))

    # 3. Content Gaps (Unique Concepts) - Visual Cards
    display(HTML("<h3 style='margin-top:35px; color:#2c3e50; font-family: sans-serif; border-bottom: 2px solid #eee; padding-bottom: 10px;'>⚠️ Content Gaps (Unique Items)</h3>"))
    u_a = result.get('unique_concepts_a', [])
    u_b = result.get('unique_concepts_b', [])

    display(HTML(f"""
        <div style="display: flex; gap: 20px; font-family: sans-serif; margin-top: 15px;">
            <div style="flex: 1; padding: 20px; background-color: #f1f8e9; border-radius: 10px; border-top: 5px solid #689f38;">
                <b style="color:#33691e; text-transform: uppercase; font-size: 12px;">Unique to Questionnaire A</b><br>
                <p style="margin-top:12px; line-height:1.6; color:#424242;">{", ".join(u_a) if u_a else "<i>No unique items identified.</i>"}</p>
            </div>
            <div style="flex: 1; padding: 20px; background-color: #fff3e0; border-radius: 10px; border-top: 5px solid #ef6c00;">
                <b style="color:#e65100; text-transform: uppercase; font-size: 12px;">Unique to Questionnaire B</b><br>
                <p style="margin-top:12px; line-height:1.6; color:#424242;">{", ".join(u_b) if u_b else "<i>No unique items identified.</i>"}</p>
            </div>
        </div>
    """))

    # 4. Detailed Qualitative Analysis
    display(HTML("<h3 style='margin-top:35px; color:#2c3e50; font-family: sans-serif; border-bottom: 2px solid #eee; padding-bottom: 10px;'>📝 LLM Qualitative Critique</h3>"))
    critique = result.get('detailed_critique', 'No qualitative analysis available.')
    display(HTML(f"""
        <div style="padding:25px; background-color:#fafafa; border: 1px dashed #cfd8dc; border-radius:10px; font-style: normal; color: #37474f; line-height: 1.8; font-family: Georgia, serif;">
            {critique}
        </div>
    """))

else:
    display(HTML(f"<div style='padding:20px; background-color:#ffebee; color:#c62828; border-radius:8px; font-weight:bold;'>❌ API Error: {result['error']}</div>"))

✅ OpenAI client initialized (gpt-5.1).
⏳ Analyzing content overlap...


,Theme (Symptom Cluster),Clinical Description,Items in Questionnaire A,Items in Questionnaire B
0,Motor vehicle and transportation accidents,Both questionnaires assess exposure to serious transportation-related accidents that could plausibly involve threat of death or serious injury.,"• A really bad car, boat, train or airplane accident",• Motor vehicle accidents
1,Other serious accidents (non-transport),Both questionnaires include exposure to serious non-transport accidents that occur in everyday environments such as work or home.,• A really bad accident at work or home,• Other accidents
2,Natural disasters,Both questionnaires cover exposure to natural disasters that can cause widespread destruction and threat to life.,"• A hurricane, flood, earthquake, tornado or fire",• Natural disasters
3,Childhood physical abuse,"Both questionnaires assess being physically hit or kicked hard enough to cause injury during childhood, consistent with clinically significant physical abuse.",• Hit or kicked hard enough to injure - as a child,• Childhood physical abuse
4,Adult physical abuse by intimate partner or others,"Both questionnaires assess being physically hit or kicked hard enough to cause injury in adulthood, which clinically overlaps with physical abuse by an intimate partner or other adults.",• Hit or kicked hard enough to injure - as an adult,• Physical abuse by an intimate partner• Severe assault by acquaintance or stranger
5,Childhood sexual abuse,"Both questionnaires assess forced or coerced sexual contact during childhood, capturing clinically similar experiences of childhood sexual abuse.",• Forced or made to have sexual contant - as a child,• Sexual abuse before age 13 by someone at least 5 year older• Sexual abuse before age 13 by someone close in age
6,Adolescent sexual abuse,"Both questionnaires include sexual victimization that occurs after childhood but before full adulthood, although Questionnaire A does not explicitly separate adolescence from adulthood.",• Forced or made to have sexual contant - as an adult,• Sexual abuse during adolescence
7,Adult sexual abuse,"Both questionnaires assess forced or coerced sexual contact in adulthood, capturing clinically similar adult sexual assault experiences.",• Forced or made to have sexual contant - as an adult,• Sexual abuse as an adult
8,Assault with a weapon,"Both questionnaires assess being attacked or robbed with a weapon, which clinically represents a high-threat interpersonal trauma.","• Attack with a gun, knife or weapon",• Robbery involving a weapon• Severe assault by acquaintance or stranger
9,Combat or warfare exposure,Both questionnaires assess exposure to warfare or combat situations that involve seeing horrific events or being in extreme danger.,• During military service - seeing something horrible or being badly scared,• Warfare or combat
